In [1]:
import os
import glob
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from PIL import Image
from tqdm import tqdm

class ViolenceDataset(Dataset):
    def __init__(self, root_dir, clip_len=8, transform=None):
        self.root_dir = root_dir
        self.clip_len = clip_len
        self.transform = transform
        self.samples = []

        for label in ["Fight", "NonFight"]:
            label_dir = os.path.join(root_dir, label)
            if not os.path.exists(label_dir):
                continue
            for video_folder in os.listdir(label_dir):
                video_path = os.path.join(label_dir, video_folder)
                frames = sorted(glob.glob(os.path.join(video_path, "*.jpg")))
                if len(frames) >= clip_len:
                    self.samples.append((frames, 0 if label == "NonFight" else 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frames, label = self.samples[idx]
        start = random.randint(0, len(frames) - self.clip_len)
        clip = frames[start:start + self.clip_len]

        imgs = []
        for f in clip:
            img = Image.open(f).convert("RGB")
            if self.transform:
                img = self.transform(img)
            imgs.append(img)

        # Stack into (C, T, H, W)
        clip_tensor = torch.stack(imgs, dim=1)
        return clip_tensor, label

def train_model(data_root, save_path="violence_detector.pth",
                epochs=5, batch_size=2, clip_len=8, lr=1e-4):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using device: {device}")

    transform = transforms.Compose([
        transforms.Resize((112, 112)),  # smaller for CPU
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_dataset = ViolenceDataset(os.path.join(data_root, "train"),
                                    clip_len=clip_len, transform=transform)
    val_dataset = ViolenceDataset(os.path.join(data_root, "val"),
                                  clip_len=clip_len, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=0)

    print(f"✅ Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    # Load pretrained model
    weights = R3D_18_Weights.DEFAULT
    model = r3d_18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_f1 = 0.0

    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        train_loss = 0.0
        all_preds, all_labels = [], []

        for clips, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            clips, labels = clips.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(clips)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(all_labels, all_preds)

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        val_preds, val_labels = [], []
        with torch.no_grad():
            for clips, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                clips, labels = clips.to(device), labels.to(device)
                outputs = model(clips)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels, val_preds)
        val_prec = precision_score(val_labels, val_preds)
        val_rec = recall_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds)

        print(f"Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {train_loss/len(train_loader):.4f} "
              f"| Train Acc: {train_acc:.4f} "
              f"| Val Loss: {val_loss/len(val_loader):.4f} "
              f"| Val Acc: {val_acc:.4f} "
              f"| Precision: {val_prec:.4f} | Recall: {val_rec:.4f} | F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)
            print(f"💾 Saved best model (epoch {epoch+1}) with F1={val_f1:.4f}")

    print(f"🏁 Training complete. Best F1: {best_f1:.4f}")



if __name__ == "__main__":
    data_root = r"C:\Users\archa\Downloads\RWF\preprocessed_frames"
    train_model(data_root,
                save_path="violence_detector.pth",
                epochs=5, batch_size=2, clip_len=8)


🚀 Using device: cpu
✅ Train samples: 1573, Val samples: 400


Epoch 1/5 [Val]: 100%|███████████████████████████████████████████████████████████████| 200/200 [08:15<00:00,  2.48s/it]


Epoch 1/5 | Train Loss: 0.6350 | Train Acc: 0.6650 | Val Loss: 0.5192 | Val Acc: 0.7625 | Precision: 0.9008 | Recall: 0.5900 | F1: 0.7130
💾 Saved best model (epoch 1) with F1=0.7130


Epoch 2/5 [Val]: 100%|███████████████████████████████████████████████████████████████| 200/200 [01:23<00:00,  2.41it/s]


Epoch 2/5 | Train Loss: 0.5341 | Train Acc: 0.7228 | Val Loss: 0.4531 | Val Acc: 0.8175 | Precision: 0.8671 | Recall: 0.7500 | F1: 0.8043
💾 Saved best model (epoch 2) with F1=0.8043


...

Epoch 3/5 | Train Loss: 0.4586 | Train Acc: 0.7788 | Val Loss: 0.5343 | Val Acc: 0.8075 | Precision: 0.8254 | Recall: 0.7800 | F1: 0.8021


Epoch 4/5 [Val]: 100%|███████████████████████████████████████████████████████████████| 200/200 [01:07<00:00,  2.95it/s]


Epoch 4/5 | Train Loss: 0.3996 | Train Acc: 0.8328 | Val Loss: 0.4930 | Val Acc: 0.8325 | Precision: 0.8182 | Recall: 0.8550 | F1: 0.8362
💾 Saved best model (epoch 4) with F1=0.8362


Epoch 5/5 [Val]: 100%|███████████████████████████████████████████████████████████████| 200/200 [01:03<00:00,  3.16it/s]

Epoch 5/5 | Train Loss: 0.3604 | Train Acc: 0.8449 | Val Loss: 0.4868 | Val Acc: 0.8125 | Precision: 0.8882 | Recall: 0.7150 | F1: 0.7922
🏁 Training complete. Best F1: 0.8362


In [1]:
import os
import glob
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights
from torchvision import transforms
from PIL import Image
import cv2
import random
import simpleaudio as sa 

def video_to_clips(video_path, clip_len=8, transform=None):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if transform:
            img = transform(img)
        frames.append(img)

    cap.release()
    clips = []
    if len(frames) >= clip_len:
        for i in range(0, len(frames) - clip_len + 1, clip_len):
            clip_frames = frames[i:i + clip_len]
            clip_tensor = torch.stack(clip_frames, dim=1)  # (C, T, H, W)
            clips.append(clip_tensor)
    return clips


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("violence_detector.pth", map_location=device))
model = model.to(device)
model.eval()


transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


alert_path = r"C:\Users\archa\Downloads\siren-alert.wav"
alert_wave = sa.WaveObject.from_wave_file(alert_path)


video_path = r"C:\Users\archa\Downloads\DLVideo\fight2.mp4"  
clips = video_to_clips(video_path, clip_len=8, transform=transform)

fight_count = 0 
alert_played = False 

for idx, clip in enumerate(clips):
    clip = clip.unsqueeze(0).to(device)  
    with torch.no_grad():
        outputs = model(clip)
        pred = torch.argmax(outputs, dim=1).item()
        if pred == 1:
            fight_count += 1
            print(f"⚠ Fight detected in clip {idx+1}! Total fight clips: {fight_count}")

            if fight_count >= 5 and not alert_played:
                print("\n🚨 ALERT! 5 fight clips detected!")
                play_obj = alert_wave.play()
                alert_played = True
        else:
            print(f"Clip {idx+1}: Non-fight")
if fight_count >= 5:
    print(f"\n🔴 Final Result: FIGHT detected ({fight_count} clips).")
else:
    print(f"\n🟢 Final Result: NO FIGHT detected ({fight_count} clips).")


Clip 1: Non-fight
Clip 2: Non-fight
Clip 3: Non-fight
Clip 4: Non-fight
Clip 5: Non-fight
Clip 6: Non-fight
⚠ Fight detected in clip 7! Total fight clips: 1
⚠ Fight detected in clip 8! Total fight clips: 2
⚠ Fight detected in clip 9! Total fight clips: 3
⚠ Fight detected in clip 10! Total fight clips: 4
⚠ Fight detected in clip 11! Total fight clips: 5

🚨 ALERT! 5 fight clips detected!
⚠ Fight detected in clip 12! Total fight clips: 6
⚠ Fight detected in clip 13! Total fight clips: 7
⚠ Fight detected in clip 14! Total fight clips: 8
⚠ Fight detected in clip 15! Total fight clips: 9
⚠ Fight detected in clip 16! Total fight clips: 10
Clip 17: Non-fight
Clip 18: Non-fight
Clip 19: Non-fight
Clip 20: Non-fight
⚠ Fight detected in clip 21! Total fight clips: 11
Clip 22: Non-fight
Clip 23: Non-fight
Clip 24: Non-fight
Clip 25: Non-fight
Clip 26: Non-fight
Clip 27: Non-fight
Clip 28: Non-fight

🔴 Final Result: FIGHT detected (11 clips).


In [ ]:
import os
import glob
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights
from torchvision import transforms
from PIL import Image
import cv2
import random
import simpleaudio as sa  

def video_to_clips(video_path, clip_len=8, transform=None):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if transform:
            img = transform(img)
        frames.append(img)

    cap.release()
    clips = []
    if len(frames) >= clip_len:
        for i in range(0, len(frames) - clip_len + 1, clip_len):
            clip_frames = frames[i:i + clip_len]
            clip_tensor = torch.stack(clip_frames, dim=1)  # (C, T, H, W)
            clips.append(clip_tensor)
    return clips


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("violence_detector.pth", map_location=device))
model = model.to(device)
model.eval()


transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


alert_path = r"C:\Users\archa\Downloads\siren-alert.wav"
alert_wave = sa.WaveObject.from_wave_file(alert_path)


video_path = r"C:\Users\archa\Downloads\RWF\RWF-2000\train\NonFight\zHRkgmIT_0.avi"
clips = video_to_clips(video_path, clip_len=8, transform=transform)

fight_count = 0  # Counter for fight clips
alert_played = False  # Ensure alert plays only once

for idx, clip in enumerate(clips):
    clip = clip.unsqueeze(0).to(device)  # Add batch dimension
    with torch.no_grad():
        outputs = model(clip)
        pred = torch.argmax(outputs, dim=1).item()
        if pred == 1:
            fight_count += 1
            print(f"⚠ Fight detected in clip {idx+1}! Total fight clips: {fight_count}")

            # Play alert immediately when 5th fight clip is detected
            if fight_count >= 5 and not alert_played:
                print("\n🚨 ALERT! 5 fight clips detected!")
                play_obj = alert_wave.play()
                alert_played = True
        else:
            print(f"Clip {idx+1}: Non-fight")
if fight_count >= 5:
    print(f"\n🔴 Final Result: FIGHT detected ({fight_count} clips).")
else:
    print(f"\n🟢 Final Result: NO FIGHT detected ({fight_count} clips).")

Clip 1: Non-fight
Clip 2: Non-fight
Clip 3: Non-fight
Clip 4: Non-fight
Clip 5: Non-fight
Clip 6: Non-fight
Clip 7: Non-fight
Clip 8: Non-fight
Clip 9: Non-fight
Clip 10: Non-fight
Clip 11: Non-fight
Clip 12: Non-fight
Clip 13: Non-fight
⚠ Fight detected in clip 14! Total fight clips: 1
Clip 15: Non-fight


In [1]:
import os
import glob
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights
from torchvision import transforms
from PIL import Image
import cv2
import random
import simpleaudio as sa  


def video_to_clips(video_path, clip_len=8, transform=None):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if transform:
            img = transform(img)
        frames.append(img)

    cap.release()
    clips = []
    if len(frames) >= clip_len:
        for i in range(0, len(frames) - clip_len + 1, clip_len):
            clip_frames = frames[i:i + clip_len]
            clip_tensor = torch.stack(clip_frames, dim=1)  # (C, T, H, W)
            clips.append(clip_tensor)
    return clips

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("violence_detector.pth", map_location=device))
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


alert_path = r"C:\Users\archa\Downloads\siren-alert.wav"
alert_wave = sa.WaveObject.from_wave_file(alert_path)


video_path = r"C:\Users\archa\Downloads\DLVideo\nofight.mp4"
clips = video_to_clips(video_path, clip_len=8, transform=transform)

fight_count = 0  
alert_played = False 

for idx, clip in enumerate(clips):
    clip = clip.unsqueeze(0).to(device)  
    with torch.no_grad():
        outputs = model(clip)
        pred = torch.argmax(outputs, dim=1).item()
        if pred == 1:
            fight_count += 1
            print(f"⚠ Fight detected in clip {idx+1}! Total fight clips: {fight_count}")

            
            if fight_count >= 5 and not alert_played:
                print("\n🚨 ALERT! 5 fight clips detected!")
                play_obj = alert_wave.play()
                alert_played = True
        else:
            print(f"Clip {idx+1}: Non-fight")
if fight_count >= 5:
    print(f"\n🔴 Final Result: FIGHT detected ({fight_count} clips).")
else:
    print(f"\n🟢 Final Result: NO FIGHT detected ({fight_count} clips).")

Clip 1: Non-fight
Clip 2: Non-fight
Clip 3: Non-fight
Clip 4: Non-fight
Clip 5: Non-fight
Clip 6: Non-fight
Clip 7: Non-fight
Clip 8: Non-fight
Clip 9: Non-fight
Clip 10: Non-fight
Clip 11: Non-fight
Clip 12: Non-fight
Clip 13: Non-fight
Clip 14: Non-fight
Clip 15: Non-fight
Clip 16: Non-fight
Clip 17: Non-fight
Clip 18: Non-fight
Clip 19: Non-fight
Clip 20: Non-fight
Clip 21: Non-fight
Clip 22: Non-fight
Clip 23: Non-fight
Clip 24: Non-fight
Clip 25: Non-fight
Clip 26: Non-fight
Clip 27: Non-fight
Clip 28: Non-fight
Clip 29: Non-fight
Clip 30: Non-fight
Clip 31: Non-fight
Clip 32: Non-fight

🟢 Final Result: NO FIGHT detected (0 clips).
